# Donor upgrade potential (expected next donation value)

**Pipeline:** Predict a supporter’s expected next donation value and explain what correlates with larger gifts.

**Data:** `supporters`, `donations` from `../Data/hiraya.db`.

**Predictive:** regression on next donation `value_php` (within a supporter timeline).

**Explanatory:** interpret associations between donation history patterns (recency, frequency, channel mix) and larger next gifts.

## 1. Problem framing

Hiraya Haven depends on donations and wants to identify supporters likely to give more if asked (`IntexContext.txt`).

- **Predictive goal:** predict a supporter’s **next donation value** (decision support for outreach prioritization).
- **Explanatory goal:** identify which donation-history patterns correlate with larger next gifts.

**Success metrics:** MAE on held-out observations; business interpretation of ranking quality (top decile vs rest).

## 2. Data acquisition, preparation & exploration

Data comes from `donations` in `../Data/hiraya.db`. We sort donations within each `supporter_id` and create a next-donation target via `shift(-1)`.

**Leakage control:** features must be computed only from information available up to the prediction date (no peeking at the next donation).

## 3. Modeling & feature selection

We use a RandomForestRegressor baseline on simple recency/frequency/proxy features plus channel/type.

## 4. Evaluation & interpretation

Report MAE. Interpret what an error of X PHP means operationally (e.g., ranking and outreach, not exact revenue accounting).

In [1]:
# ============================================================================
# Donor upgrade — Predict next gift size from current gift history
# ============================================================================
# Notebook code cell overview — see inline comments below.
#
# Rows are donations in time order; target `next_value_php` is the following gift for that supporter.
# Features include current value, donation index (sequence), days since previous gift, recurring flag.
# Categoricals (donation_type, channel) are dummy-encoded; drop_first avoids full rank collinearity.
# GroupShuffleSplit by supporter_id ensures the same donor is not in train AND test — critical
#   because otherwise the model memorizes donor-specific patterns and metrics look too good.
# RandomForest gives nonlinear effects + default feature importance for ranking drivers.
#
from __future__ import annotations

import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupShuffleSplit

sns.set_theme(style="whitegrid", context="notebook")

ROOT = Path("..").resolve()
DB_PATH = ROOT / "Data" / "hiraya.db"

with sqlite3.connect(DB_PATH) as conn:
    supporters = pd.read_sql_query('SELECT * FROM "supporters"', conn)
    donations = pd.read_sql_query('SELECT * FROM "donations"', conn)

donations["donation_date"] = pd.to_datetime(donations["donation_date"], errors="coerce")

# Define donation value
val = donations["amount"].fillna(donations["estimated_value"]).fillna(0.0)
donations["value_php"] = val

# Sort donations within supporter and create "next donation" target
donations = donations.sort_values(["supporter_id", "donation_date"]).copy()
g = donations.groupby("supporter_id", group_keys=False)
donations["next_value_php"] = g["value_php"].shift(-1)
donations["days_since_prev"] = (donations["donation_date"] - g["donation_date"].shift(1)).dt.days

# History features up to this donation
# (simple baseline: recency and current donation attributes)
donations["donation_index"] = g.cumcount() + 1

# Use rows where next_value exists
train_df = donations.dropna(subset=["next_value_php"]).copy()
train_df["days_since_prev"] = train_df["days_since_prev"].fillna(train_df["days_since_prev"].median())

features = [
    "value_php",
    "donation_index",
    "days_since_prev",
    "is_recurring",
]

# Encode donation_type and channel_source as categories
train_df["donation_type"] = train_df["donation_type"].fillna("(unknown)")
train_df["channel_source"] = train_df["channel_source"].fillna("(unknown)")

X = pd.get_dummies(train_df[features + ["donation_type", "channel_source"]], drop_first=True)
y = train_df["next_value_php"].astype(float)

# IMPORTANT evaluation logic (Ch. 15):
# Split by supporter_id so the same supporter does not appear in both train and test.
# This prevents optimistic leakage from supporter-specific donation patterns.
groups = train_df["supporter_id"].values
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

rf = RandomForestRegressor(n_estimators=400, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
pred = rf.predict(X_test)

print("MAE (next donation value, grouped by supporter):", round(mean_absolute_error(y_test, pred), 2))

imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
imp.head(15)

MAE (next donation value, grouped by supporter): 679.98


value_php                         0.388898
days_since_prev                   0.249458
donation_index                    0.174706
channel_source_Event              0.035852
channel_source_Direct             0.034369
donation_type_Monetary            0.029197
is_recurring                      0.028404
channel_source_SocialMedia        0.020475
channel_source_PartnerReferral    0.015580
donation_type_Skills              0.009638
donation_type_Time                0.009468
donation_type_SocialMedia         0.003954
dtype: float64

## Notes on interpretation, causality, and deployment

- **Predictive use:** rank supporters by expected next donation value to prioritize outreach.
- **Explanatory use:** treat feature importances as *what the model used*, not proof of causality.

### Leakage caution
If you deploy this in production, be careful to compute features using only information available **before** the prediction date.

### Deployment notes
- Produce a monthly batch table: `supporter_upgrade_scores(supporter_id, score_date, predicted_next_value_php)`.
- Show top candidates in the admin dashboard with “why” (top drivers / recent history summary).